In [1]:
import numpy as np
import pandas as pd
import os
import pickle
import gc
from datetime import datetime


In [2]:
DATA_DIR = r"C:\MATH699P\Data"
PROCESSED_DIR = os.path.join(DATA_DIR, 'processed_data')
MODEL_DATA_DIR = os.path.join(PROCESSED_DIR, 'model_ready_dask')  

SEQ_DIR = os.path.join(MODEL_DATA_DIR, 'sequences')
SEQ_TRAIN_DIR = os.path.join(SEQ_DIR, 'train')
SEQ_VAL_DIR = os.path.join(SEQ_DIR, 'val')
SEQ_TEST_DIR = os.path.join(SEQ_DIR, 'test')

for d in [SEQ_DIR, SEQ_TRAIN_DIR, SEQ_VAL_DIR, SEQ_TEST_DIR]:
    os.makedirs(d, exist_ok=True)

with open(os.path.join(MODEL_DATA_DIR, 'preprocessing.pkl'), 'rb') as f:
    preprocessing = pickle.load(f)

FEATURE_COLS = preprocessing['feature_cols']
TARGET_COL = preprocessing['target_col']
CONFIG = preprocessing['config']

SEQ_LENGTH = CONFIG['sequence_length']  # 24
FORECAST_HORIZON = CONFIG['forecast_horizon']  # 1

print(f"Features: {len(FEATURE_COLS)}")
print(f"Sequence length: {SEQ_LENGTH}")
print(f"Forecast horizon: {FORECAST_HORIZON}")

Features: 67
Sequence length: 24
Forecast horizon: 1


SECTION 1: CREATING SEQUENCES PER-SITE

In [3]:
def create_and_save_sequences_for_split(parquet_dir, output_dir, feature_cols, 
                                         target_col, seq_length, forecast_horizon,
                                         preprocessing):
    """
    Create sequences and save EACH SITE to a separate file.
    """
    for f in os.listdir(output_dir):
        if f.endswith('.npz'):
            os.remove(os.path.join(output_dir, f))
    
    df = pd.read_parquet(parquet_dir)
    print(f"Loaded {len(df):,} rows")
    print(f"Columns: {df.columns.tolist()[:10]}...")  
    
    if 'SITE_ID' not in df.columns:
        print("ERROR: SITE_ID not in dataframe!")
        print(f"Available columns: {df.columns.tolist()}")
        return 0, 0
    
    sites = df['SITE_ID'].unique()
    print(f"Processing {len(sites)} sites...")
    
    total_sequences = 0
    sites_with_sequences = 0
    
    for i, site_id in enumerate(sites):
        if (i + 1) % 20 == 0:
            print(f"{i+1}/{len(sites)} sites, {total_sequences:,} sequences so far...")
        
        site_df = df[df['SITE_ID'] == site_id].sort_values('DATE_TIME').reset_index(drop=True)
        
        if len(site_df) < seq_length + forecast_horizon:
            continue
        
        features = site_df[feature_cols].values.astype('float32')
        target = site_df[target_col].values.astype('float32')
        
        n_seq = len(site_df) - seq_length - forecast_horizon + 1
        
        X_site = np.zeros((n_seq, seq_length, len(feature_cols)), dtype='float32')
        y_site = np.zeros(n_seq, dtype='float32')
        
        for j in range(n_seq):
            X_site[j] = features[j:j + seq_length]
            y_site[j] = target[j + seq_length + forecast_horizon - 1]
        
        np.savez_compressed(
            os.path.join(output_dir, f'{site_id}.npz'),
            X=X_site, y=y_site
        )
        
        total_sequences += n_seq
        sites_with_sequences += 1
        
        del site_df, features, target, X_site, y_site
    
    del df
    gc.collect()
    
    print(f"DONE: {total_sequences:,} sequences from {sites_with_sequences} sites")
    return total_sequences, sites_with_sequences

In [4]:
print("CREATING TRAINING SEQUENCES")
print("=" * 80)

train_parquet = os.path.join(MODEL_DATA_DIR, 'train')
train_seqs, train_sites = create_and_save_sequences_for_split(
    train_parquet, SEQ_TRAIN_DIR, FEATURE_COLS, TARGET_COL, SEQ_LENGTH, FORECAST_HORIZON, preprocessing
)
gc.collect()

CREATING TRAINING SEQUENCES
Loaded 18,193,749 rows
Columns: ['SITE_ID', 'DATE_TIME', 'OZONE', 'year', 'month', 'day', 'hour', 'dayofweek', 'dayofyear', 'week']...
Processing 123 sites...
20/123 sites, 2,831,516 sequences so far...
40/123 sites, 6,386,662 sequences so far...
60/123 sites, 9,169,728 sequences so far...
80/123 sites, 11,570,157 sequences so far...
100/123 sites, 15,097,032 sequences so far...
120/123 sites, 17,784,910 sequences so far...
DONE: 18,190,797 sequences from 123 sites


0

In [5]:
print("\nCREATING VALIDATION SEQUENCES")
print("=" * 80)

val_parquet = os.path.join(MODEL_DATA_DIR, 'val')
val_seqs, val_sites = create_and_save_sequences_for_split(
    val_parquet, SEQ_VAL_DIR, FEATURE_COLS, TARGET_COL, SEQ_LENGTH, FORECAST_HORIZON, preprocessing
)
gc.collect()


CREATING VALIDATION SEQUENCES
Loaded 2,072,797 rows
Columns: ['SITE_ID', 'DATE_TIME', 'OZONE', 'year', 'month', 'day', 'hour', 'dayofweek', 'dayofyear', 'week']...
Processing 89 sites...
20/89 sites, 449,742 sequences so far...
40/89 sites, 904,477 sequences so far...
60/89 sites, 1,388,130 sequences so far...
80/89 sites, 1,841,558 sequences so far...
DONE: 2,070,661 sequences from 89 sites


0

In [6]:
print("\nCREATING TEST SEQUENCES")
print("=" * 80)

test_parquet = os.path.join(MODEL_DATA_DIR, 'test')
test_seqs, test_sites = create_and_save_sequences_for_split(
    test_parquet, SEQ_TEST_DIR, FEATURE_COLS, TARGET_COL, SEQ_LENGTH, FORECAST_HORIZON, preprocessing
)
gc.collect()


CREATING TEST SEQUENCES
Loaded 1,400,484 rows
Columns: ['SITE_ID', 'DATE_TIME', 'OZONE', 'year', 'month', 'day', 'hour', 'dayofweek', 'dayofyear', 'week']...
Processing 84 sites...
20/84 sites, 316,998 sequences so far...
40/84 sites, 645,770 sequences so far...
60/84 sites, 993,773 sequences so far...
80/84 sites, 1,332,082 sequences so far...
DONE: 1,398,469 sequences from 83 sites


0

SECTION 2: DATA GENERATOR

In [7]:
class SequenceGenerator:
    """
    Memory-efficient data generator that loads sequences from disk on-the-fly.
    Compatible with Keras/TensorFlow and PyTorch.
    """
    
    def __init__(self, seq_dir, batch_size=1024, shuffle=True):
        self.seq_dir = seq_dir
        self.batch_size = batch_size
        self.shuffle = shuffle
        
        self.files = [os.path.join(seq_dir, f) for f in os.listdir(seq_dir) if f.endswith('.npz')]
        
        self.total_sequences = 0
        self.file_sizes = []
        for f in self.files:
            data = np.load(f)
            n = len(data['y'])
            self.file_sizes.append(n)
            self.total_sequences += n
        
        print(f"Generator initialized: {self.total_sequences:,} sequences from {len(self.files)} files")
    
    def __len__(self):
        """Number of batches per epoch."""
        return int(np.ceil(self.total_sequences / self.batch_size))
    
    def generate(self):
        """
        Generator that yields (X_batch, y_batch) tuples.
        Use this with model.fit(generator.generate(), steps_per_epoch=len(generator))
        """
        while True:  
            files = self.files.copy()
            if self.shuffle:
                np.random.shuffle(files)
            
            X_buffer = []
            y_buffer = []
            buffer_size = 0
            
            for f in files:
                data = np.load(f)
                X_site = data['X']
                y_site = data['y']
                
                X_buffer.append(X_site)
                y_buffer.append(y_site)
                buffer_size += len(y_site)
                
                while buffer_size >= self.batch_size:
                    X_all = np.concatenate(X_buffer, axis=0)
                    y_all = np.concatenate(y_buffer, axis=0)
                    
                    if self.shuffle:
                        idx = np.random.permutation(len(y_all))
                        X_all = X_all[idx]
                        y_all = y_all[idx]
                    
                    yield X_all[:self.batch_size], y_all[:self.batch_size]
                    
                    X_buffer = [X_all[self.batch_size:]]
                    y_buffer = [y_all[self.batch_size:]]
                    buffer_size = len(y_buffer[0])
            
            if buffer_size > 0:
                X_all = np.concatenate(X_buffer, axis=0)
                y_all = np.concatenate(y_buffer, axis=0)
                yield X_all, y_all
    
    def get_sample_batch(self):
        """Get a single batch for testing/debugging."""
        gen = self.generate()
        return next(gen)

In [8]:
print("\nTesting generator...")

train_gen = SequenceGenerator(SEQ_TRAIN_DIR, batch_size=1024)
X_batch, y_batch = train_gen.get_sample_batch()

print(f"\nSample batch:")
print(f"X_batch shape: {X_batch.shape}")
print(f"y_batch shape: {y_batch.shape}")
print(f"X_batch dtype: {X_batch.dtype}")
print(f"y_batch dtype: {y_batch.dtype}")


Testing generator...
Generator initialized: 18,190,797 sequences from 123 files

Sample batch:
X_batch shape: (1024, 24, 67)
y_batch shape: (1024,)
X_batch dtype: float32
y_batch dtype: float32


SECTION 3: SAVE GENERATOR INFO FOR TRAINING

In [9]:
seq_info = {
    'train_sequences': train_seqs,
    'val_sequences': val_seqs,
    'test_sequences': test_seqs,
    'train_sites': train_sites,
    'val_sites': val_sites,
    'test_sites': test_sites,
    'sequence_length': SEQ_LENGTH,
    'n_features': len(FEATURE_COLS),
    'forecast_horizon': FORECAST_HORIZON,
    'seq_train_dir': SEQ_TRAIN_DIR,
    'seq_val_dir': SEQ_VAL_DIR,
    'seq_test_dir': SEQ_TEST_DIR,
}

with open(os.path.join(MODEL_DATA_DIR, 'sequence_info.pkl'), 'wb') as f:
    pickle.dump(seq_info, f)

print("Saved sequence_info.pkl")

Saved sequence_info.pkl


SECTION 4: SUMMARY

In [10]:
print("SEQUENCE CREATION COMPLETE")
print("=" * 80)

print(f"\nSEQUENCES CREATED:")
print(f"Train: {train_seqs:,} sequences from {train_sites} sites")
print(f"Val:   {val_seqs:,} sequences from {val_sites} sites")
print(f"Test:  {test_seqs:,} sequences from {test_sites} sites")

print(f"\nSEQUENCE SHAPE: ({SEQ_LENGTH}, {len(FEATURE_COLS)})")

print(f"\nFILES SAVED:")
print(f"{SEQ_TRAIN_DIR}/ ({len(os.listdir(SEQ_TRAIN_DIR))} files)")
print(f"{SEQ_VAL_DIR}/ ({len(os.listdir(SEQ_VAL_DIR))} files)")
print(f"{SEQ_TEST_DIR}/ ({len(os.listdir(SEQ_TEST_DIR))} files)")

total_size = 0
for d in [SEQ_TRAIN_DIR, SEQ_VAL_DIR, SEQ_TEST_DIR]:
    for f in os.listdir(d):
        total_size += os.path.getsize(os.path.join(d, f))
print(f"\nTOTAL DISK SPACE: {total_size / 1e9:.2f} GB")

SEQUENCE CREATION COMPLETE

SEQUENCES CREATED:
Train: 18,190,797 sequences from 123 sites
Val:   2,070,661 sequences from 89 sites
Test:  1,398,469 sequences from 83 sites

SEQUENCE SHAPE: (24, 67)

FILES SAVED:
C:\MATH699P\Data\processed_data\model_ready_dask\sequences\train/ (123 files)
C:\MATH699P\Data\processed_data\model_ready_dask\sequences\val/ (89 files)
C:\MATH699P\Data\processed_data\model_ready_dask\sequences\test/ (83 files)

TOTAL DISK SPACE: 3.33 GB
